In [1]:
import pyulog
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")  # non-interactive backend for script execution
import matplotlib.pyplot as plt
from pathlib import Path

LOG_ROOT = Path("../px4_logs")

In [2]:
ulg_files = sorted(LOG_ROOT.rglob("*.ulg"))
print(f"Found {len(ulg_files)} .ulg files")

rows = []
for p in ulg_files:
    try:
        log = pyulog.ULog(str(p), disable_str_exceptions=True)
        topics = {d.name for d in log.data_list}
        rows.append({"path": str(p.relative_to(LOG_ROOT)), "topics": len(topics),
                     "has_actuator": "actuator_outputs" in topics,
                     "has_imu": "sensor_combined" in topics,
                     "has_rates": "vehicle_angular_velocity" in topics,
                     "has_position": "vehicle_local_position" in topics,
                     "has_attitude": "vehicle_attitude" in topics,
                     "has_rate_sp": "vehicle_rates_setpoint" in topics,
                     "has_pos_sp": "vehicle_local_position_setpoint" in topics,
                     "duration_s": (log.last_timestamp - log.start_timestamp) / 1e6})
    except Exception as e:
        rows.append({"path": str(p.relative_to(LOG_ROOT)), "error": str(e)})

df = pd.DataFrame(rows)
df

Found 126 .ulg files


,path,topics,has_actuator,has_imu,has_rates,has_position,has_attitude,has_rate_sp,has_pos_sp,duration_s
0,Hermit/07_09_2024/log_0_2024-9-7-15-13-48.ulg,70,True,True,True,True,True,True,False,98.187965
1,Hermit/07_09_2024/log_1_2024-9-7-15-19-36.ulg,67,True,True,True,True,True,True,False,121.843157
2,Hermit/07_09_2024/log_2_2024-9-7-15-23-14.ulg,67,True,True,True,True,True,True,False,27.834936
3,Hermit/07_09_2024/log_3_2024-9-7-15-24-34.ulg,66,True,True,True,True,True,True,False,32.029712
4,Hermit/07_09_2024/log_4_2024-9-7-15-34-16.ulg,74,True,True,True,True,True,True,True,8.520989
...,...,...,...,...,...,...,...,...,...,...
121,pixhawk_2.4.8/5-06/log_5_2024-6-5-18-34-46.ulg,60,True,True,True,True,True,True,True,19.198301
122,pixhawk_2.4.8/5-06/log_6_2024-6-5-18-36-12.ulg,61,True,True,True,True,True,True,True,73.597680
123,pixhawk_2.4.8/5-06/log_7_2024-6-5-18-38-10.ulg,60,True,True,True,True,True,True,True,3.811800
124,pixhawk_2.4.8/5-06/log_8_2024-6-5-18-38-52.ulg,57,True,True,True,True,True,True,False,33.079042


In [3]:
bool_cols = ["has_actuator", "has_imu", "has_rates", "has_position",
             "has_attitude", "has_rate_sp", "has_pos_sp"]
heat = df[bool_cols].fillna(False).astype(float)
fig, ax = plt.subplots(figsize=(10, max(4, len(heat) * 0.3)))
im = ax.imshow(heat.T, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_yticks(range(len(bool_cols))); ax.set_yticklabels(bool_cols)
ax.set_xlabel("Log index"); ax.set_title("Signal availability across all .ulg files")
plt.tight_layout(); plt.savefig("/tmp/heatmap.png"); plt.close()
print("Saved heatmap to /tmp/heatmap.png")

Saved heatmap to /tmp/heatmap.png


In [4]:
LOG_PATH = "../px4_logs/Hermit/testes_PID_position_31-08/log_12_2024-8-30-16-21-54.ulg"
log = pyulog.ULog(LOG_PATH, disable_str_exceptions=True)
t0 = log.start_timestamp

def to_df(log, topic, multi_id=0):
    d = next(d for d in log.data_list if d.name == topic and d.multi_id == multi_id)
    df = pd.DataFrame({f: d.data[f] for f in d.data})
    df["t_s"] = (df["timestamp"] - t0) / 1e6
    return df

fig, axes = plt.subplots(4, 1, figsize=(14, 12), sharex=True)

# Actuator outputs
try:
    act = to_df(log, "actuator_outputs")
    for i in range(4):
        axes[0].plot(act["t_s"], act[f"output[{i}]"], label=f"motor {i+1}")
    axes[0].set_title("Actuator Outputs"); axes[0].legend()
except Exception as e:
    axes[0].set_title(f"Actuator Outputs (unavailable: {e})")

# Angular velocity
try:
    rates = to_df(log, "vehicle_angular_velocity")
    for ax_name, col in [("roll", "xyz[0]"), ("pitch", "xyz[1]"), ("yaw", "xyz[2]")]:
        axes[1].plot(rates["t_s"], np.rad2deg(rates[col]), label=ax_name)
    axes[1].set_title("Angular Velocity (deg/s)"); axes[1].legend()
except Exception as e:
    axes[1].set_title(f"Angular Velocity (unavailable: {e})")

# Attitude
try:
    att = to_df(log, "vehicle_attitude")
    from scipy.spatial.transform import Rotation as R_
    q = att[["q[0]", "q[1]", "q[2]", "q[3]"]].values  # [w, x, y, z]
    euler = R_.from_quat(q[:, [1, 2, 3, 0]]).as_euler("xyz", degrees=True)
    for i, name in enumerate(["roll", "pitch", "yaw"]):
        axes[2].plot(att["t_s"].values, euler[:, i], label=name)
    axes[2].set_title("Attitude (deg)"); axes[2].legend()
except Exception as e:
    axes[2].set_title(f"Attitude (unavailable: {e})")

# Local position
try:
    pos = to_df(log, "vehicle_local_position")
    for col, name in [("x", "X"), ("y", "Y"), ("z", "Z")]:
        axes[3].plot(pos["t_s"], pos[col], label=name)
    axes[3].set_title("Local Position (m)"); axes[3].legend(); axes[3].set_xlabel("Time (s)")
except Exception as e:
    axes[3].set_title(f"Local Position (unavailable: {e})"); axes[3].set_xlabel("Time (s)")

plt.tight_layout(); plt.savefig("/tmp/signals.png"); plt.close()
print("Saved signal plot to /tmp/signals.png")

Saved signal plot to /tmp/signals.png


In [5]:
print("=== PID Parameters from log ===")
pid_params = {k: v for k, v in log.initial_parameters.items()
              if any(x in k for x in ["MC_ROLL", "MC_PITCH", "MC_YAW", "MPC_XY", "MPC_Z_"])}
for k in sorted(pid_params):
    print(f"  {k}: {pid_params[k]:.6f}")

=== PID Parameters from log ===
  MC_PITCHRATE_D: 0.002200
  MC_PITCHRATE_FF: 0.000000
  MC_PITCHRATE_I: 0.175000
  MC_PITCHRATE_K: 0.450000
  MC_PITCHRATE_MAX: 220.000000
  MC_PITCHRATE_P: 0.216778
  MC_PITCH_P: 7.000000
  MC_ROLLRATE_D: 0.002200
  MC_ROLLRATE_FF: 0.000000
  MC_ROLLRATE_I: 0.175000
  MC_ROLLRATE_K: 0.450000
  MC_ROLLRATE_MAX: 220.000000
  MC_ROLLRATE_P: 0.215126
  MC_ROLL_P: 7.000000
  MC_YAWRATE_D: 0.003545
  MC_YAWRATE_FF: 0.000000
  MC_YAWRATE_I: 0.160000
  MC_YAWRATE_K: 1.750000
  MC_YAWRATE_MAX: 200.000000
  MC_YAWRATE_P: 0.154420
  MC_YAW_P: 5.000000
  MC_YAW_WEIGHT: 0.400000
  MPC_XY_CRUISE: 5.000000
  MPC_XY_ERR_MAX: 2.000000
  MPC_XY_MAN_EXPO: 0.600000
  MPC_XY_P: 1.100000
  MPC_XY_TRAJ_P: 0.500000
  MPC_XY_VEL_ALL: -10.000000
  MPC_XY_VEL_D_ACC: 0.200000
  MPC_XY_VEL_I_ACC: 3.000000
  MPC_XY_VEL_MAX: 12.000000
  MPC_XY_VEL_P_ACC: 2.050000
  MPC_Z_MAN_EXPO: 0.600000
  MPC_Z_P: 1.000000
  MPC_Z_VEL_ALL: -3.000000
  MPC_Z_VEL_D_ACC: 0.000000
  MPC_Z_VEL_I_ACC: 